# Task 5 – Loan Default Prediction

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

from sklearn.model_selection import cross_validate, RandomizedSearchCV

import warnings
warnings.filterwarnings("ignore")

### Load Dataset

In [2]:
df = pd.read_csv("Loan_default.csv")

print(df.head())
print(df.shape)

       LoanID  Age  Income  LoanAmount  CreditScore  MonthsEmployed  \
0  I38PQUQS96   56   85994       50587          520              80   
1  HPSK72WA7R   69   50432      124440          458              15   
2  C1OZ6DPJ8Y   46   84208      129188          451              26   
3  V2KKSFM3UN   32   31713       44799          743               0   
4  EY08JDHTZP   60   20437        9139          633               8   

   NumCreditLines  InterestRate  LoanTerm  DTIRatio    Education  \
0               4         15.23        36      0.44   Bachelor's   
1               1          4.81        60      0.68     Master's   
2               3         21.17        24      0.31     Master's   
3               3          7.07        24      0.23  High School   
4               4          6.51        48      0.73   Bachelor's   

  EmploymentType MaritalStatus HasMortgage HasDependents LoanPurpose  \
0      Full-time      Divorced         Yes           Yes       Other   
1      Full-time    

In [3]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 255347 entries, 0 to 255346
Data columns (total 18 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   LoanID          255347 non-null  object 
 1   Age             255347 non-null  int64  
 2   Income          255347 non-null  int64  
 3   LoanAmount      255347 non-null  int64  
 4   CreditScore     255347 non-null  int64  
 5   MonthsEmployed  255347 non-null  int64  
 6   NumCreditLines  255347 non-null  int64  
 7   InterestRate    255347 non-null  float64
 8   LoanTerm        255347 non-null  int64  
 9   DTIRatio        255347 non-null  float64
 10  Education       255347 non-null  object 
 11  EmploymentType  255347 non-null  object 
 12  MaritalStatus   255347 non-null  object 
 13  HasMortgage     255347 non-null  object 
 14  HasDependents   255347 non-null  object 
 15  LoanPurpose     255347 non-null  object 
 16  HasCoSigner     255347 non-null  object 
 17  Default   

In [4]:
print("Dataset Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nMissing Values:")
print(df.isnull().sum())

print("\nTarget Distribution:")
print(df["Default"].value_counts())

Dataset Shape: (255347, 18)

Columns:
['LoanID', 'Age', 'Income', 'LoanAmount', 'CreditScore', 'MonthsEmployed', 'NumCreditLines', 'InterestRate', 'LoanTerm', 'DTIRatio', 'Education', 'EmploymentType', 'MaritalStatus', 'HasMortgage', 'HasDependents', 'LoanPurpose', 'HasCoSigner', 'Default']

Missing Values:
LoanID            0
Age               0
Income            0
LoanAmount        0
CreditScore       0
MonthsEmployed    0
NumCreditLines    0
InterestRate      0
LoanTerm          0
DTIRatio          0
Education         0
EmploymentType    0
MaritalStatus     0
HasMortgage       0
HasDependents     0
LoanPurpose       0
HasCoSigner       0
Default           0
dtype: int64

Target Distribution:
Default
0    225694
1     29653
Name: count, dtype: int64


### 1.Separate features and target


In [5]:
X = df.drop(columns=["Default", "LoanID"])
y = df["Default"]

print(X.shape)
print(y.shape)

(255347, 16)
(255347,)


### 2.Check the target distribution


In [6]:
print(y.value_counts())
print(y.value_counts(normalize=True) * 100)

Default
0    225694
1     29653
Name: count, dtype: int64
Default
0    88.387175
1    11.612825
Name: proportion, dtype: float64


### 3.Identify numerical and categorical columns


In [7]:
categorical_cols = X.select_dtypes(include=["object"]).columns
numerical_cols = X.select_dtypes(exclude=["object"]).columns

print("Categorical columns:")
print(categorical_cols)

print("\nNumerical columns:")
print(numerical_cols)


Categorical columns:
Index(['Education', 'EmploymentType', 'MaritalStatus', 'HasMortgage',
       'HasDependents', 'LoanPurpose', 'HasCoSigner'],
      dtype='object')

Numerical columns:
Index(['Age', 'Income', 'LoanAmount', 'CreditScore', 'MonthsEmployed',
       'NumCreditLines', 'InterestRate', 'LoanTerm', 'DTIRatio'],
      dtype='object')


### 4.Preprocess the data


In [8]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
    ]
)

### 5.Split the data into training and testing data

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

Training data: (204277, 16)
Testing data: (51070, 16)


### 6.Create a function for model evaluation


In [10]:
def evaluate_model(model, X_train, X_test, y_train, y_test):

    model.fit(X_train, y_train)

    train_pred = model.predict(X_train)
    test_pred = model.predict(X_test)

    print("Train Accuracy :", accuracy_score(y_train, train_pred))
    print("Test Accuracy  :", accuracy_score(y_test, test_pred))

    print("Test Precision :", precision_score(y_test, test_pred))
    print("Test Recall    :", recall_score(y_test, test_pred))
    print("Test F1-score  :", f1_score(y_test, test_pred))

### 7.Logistic Regression


In [11]:
logistic_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=1000))
])

evaluate_model(
    logistic_model,
    X_train,
    X_test,
    y_train,
    y_test
)

Train Accuracy : 0.885102091767551
Test Accuracy  : 0.885275112590562
Test Precision : 0.608433734939759
Test Recall    : 0.03405833754847412
Test F1-score  : 0.06450582787801373


### 8.Decision Tree


In [12]:
decision_tree = Pipeline([
    ("preprocessor", preprocessor),
    ("model", DecisionTreeClassifier(random_state=42))
])

evaluate_model(
    decision_tree,
    X_train,
    X_test,
    y_train,
    y_test
)

Train Accuracy : 1.0
Test Accuracy  : 0.8016056393185823
Test Precision : 0.19755219582433406
Test Recall    : 0.23132692631933907
Test F1-score  : 0.21310966138552345


### 9.Random Forest


In [14]:
random_forest = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    ))
])

evaluate_model(
    random_forest,
    X_train,
    X_test,
    y_train,
    y_test
)

Train Accuracy : 0.9999706281177029
Test Accuracy  : 0.885275112590562
Test Precision : 0.6333333333333333
Test Recall    : 0.028831562974203338
Test F1-score  : 0.055152394775036286


### 10.AdaBoost


In [15]:
adaboost = Pipeline([
    ("preprocessor", preprocessor),
    ("model", AdaBoostClassifier(
        n_estimators=100,
        random_state=42
    ))
])

evaluate_model(
    adaboost,
    X_train,
    X_test,
    y_train,
    y_test
)

Train Accuracy : 0.885787435687816
Test Accuracy  : 0.8858429606422558
Test Precision : 0.6234718826405868
Test Recall    : 0.042994436014162876
Test F1-score  : 0.0804416403785489


### 11.Gradient Boosting


In [16]:
gradient_boosting = Pipeline([
    ("preprocessor", preprocessor),
    ("model", GradientBoostingClassifier(
        n_estimators=100,
        random_state=42
    ))
])

evaluate_model(
    gradient_boosting,
    X_train,
    X_test,
    y_train,
    y_test
)

Train Accuracy : 0.8867273359213225
Test Accuracy  : 0.8861758370863521
Test Precision : 0.6260683760683761
Test Recall    : 0.049401450008430284
Test F1-score  : 0.09157680887638693


### 12.Compare all models


In [18]:
models = {
    "Logistic Regression": logistic_model,
    "Decision Tree": decision_tree,
    "Random Forest": random_forest,
    "AdaBoost": adaboost,
    "Gradient Boosting": gradient_boosting
}

results = []

for name, model in models.items():

    model.fit(X_train, y_train)

    train_pred = model.predict(X_train)
    test_pred = model.predict(X_test)

    results.append({
        "Model": name,
        "Train Accuracy": accuracy_score(y_train, train_pred),
        "Test Accuracy": accuracy_score(y_test, test_pred),
        "Precision": precision_score(y_test, test_pred),
        "Recall": recall_score(y_test, test_pred),
        "F1-score": f1_score(y_test, test_pred)
    })

results_df = pd.DataFrame(results)

print(results_df)

                 Model  Train Accuracy  Test Accuracy  Precision    Recall  \
0  Logistic Regression        0.885102       0.885275   0.608434  0.034058   
1        Decision Tree        1.000000       0.801606   0.197552  0.231327   
2        Random Forest        0.999971       0.885275   0.633333  0.028832   
3             AdaBoost        0.885787       0.885843   0.623472  0.042994   
4    Gradient Boosting        0.886727       0.886176   0.626068  0.049401   

   F1-score  
0  0.064506  
1  0.213110  
2  0.055152  
3  0.080442  
4  0.091577  


### 13.Check Overfitting / Underfitting


In [19]:
print(results_df[
    ["Model", "Train Accuracy", "Test Accuracy"]
])

                 Model  Train Accuracy  Test Accuracy
0  Logistic Regression        0.885102       0.885275
1        Decision Tree        1.000000       0.801606
2        Random Forest        0.999971       0.885275
3             AdaBoost        0.885787       0.885843
4    Gradient Boosting        0.886727       0.886176


### 14.Perform 5-Fold Cross-Validation


In [20]:
cv_results = []

for name, model in models.items():

    scores = cross_validate(
        model,
        X_train,
        y_train,
        cv=5,
        scoring="f1",
        n_jobs=-1
    )

    cv_results.append({
        "Model": name,
        "Mean F1": scores["test_score"].mean(),
        "F1 Std": scores["test_score"].std()
    })

cv_df = pd.DataFrame(cv_results)

print(cv_df)

                 Model   Mean F1    F1 Std
0  Logistic Regression  0.064208  0.002691
1        Decision Tree  0.211359  0.004838
2        Random Forest  0.054339  0.003728
3             AdaBoost  0.088891  0.005079
4    Gradient Boosting  0.091856  0.002727


### 15.Select the best model


In [21]:
best_model_name = cv_df.loc[
    cv_df["Mean F1"].idxmax(),
    "Model"
]

print("Best Model:", best_model_name)

Best Model: Decision Tree


### 16.Perform Hyperparameter Tuning


In [22]:
param_grid = {
    "model__max_depth": [5, 10, 15, 20, None],
    "model__min_samples_split": [2, 5, 10],
    "model__min_samples_leaf": [1, 2, 4]
}

### 17.Run RandomizedSearchCV

In [23]:
random_search = RandomizedSearchCV(
    decision_tree,
    param_distributions=param_grid,
    n_iter=10,
    cv=5,
    scoring="f1",
    random_state=42,
    n_jobs=-1
)

random_search.fit(X_train, y_train)

RandomizedSearchCV(cv=5,
                   estimator=Pipeline(steps=[('preprocessor',
                                              ColumnTransformer(transformers=[('num',
                                                                               StandardScaler(),
                                                                               Index(['Age', 'Income', 'LoanAmount', 'CreditScore', 'MonthsEmployed',
       'NumCreditLines', 'InterestRate', 'LoanTerm', 'DTIRatio'],
      dtype='object')),
                                                                              ('cat',
                                                                               OneHotEncoder(handle_unknown='ignore'),
                                                                               Index(['Education', 'EmploymentType', 'MaritalStatus', 'HasMortgage',
       'HasDependents', 'LoanPurpose', 'HasCoSigner'],
      dtype='object'))])),
                                             ('model',
                                              DecisionTreeClassifier(random_state=42))]),
                   n_jobs=-1,
                   param_distributions={'model__max_depth': [5, 10, 15, 20,
                                                             None],
                                        'model__min_samples_leaf': [1, 2, 4],
                                        'model__min_samples_split': [2, 5, 10]},
                   random_state=42, scoring='f1')

### 18.Find the best parameters


In [24]:
print("Best Parameters:")
print(random_search.best_params_)

print("\nBest CV F1-score:")
print(random_search.best_score_)

Best Parameters:
{'model__min_samples_split': 10, 'model__min_samples_leaf': 2, 'model__max_depth': None}

Best CV F1-score:
0.20304295776998252


### 19.Test the tuned model


In [25]:
tuned_model = random_search.best_estimator_

y_pred = tuned_model.predict(X_test)

print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1-score :", f1_score(y_test, y_pred))

Accuracy : 0.8302525944781672
Precision: 0.22264991896272285
Recall   : 0.18529758893947057
F1-score : 0.20226373424128094


### 20.Compare Before vs After Tuning


In [26]:
baseline_pred = decision_tree.predict(X_test)

baseline_f1 = f1_score(y_test, baseline_pred)
tuned_f1 = f1_score(y_test, y_pred)

print("Baseline F1-score:", baseline_f1)
print("Tuned F1-score   :", tuned_f1)

Baseline F1-score: 0.21310966138552345
Tuned F1-score   : 0.20226373424128094
